# 02 — Classificação Semi-Automática dos Vídeos do Cliente

Classifica cada segmento dos vídeos do cliente como `violence` ou `non_violence`,  
usando detecção automática + revisão manual dos casos incertos.

**Pipeline:**
1. Segmentação: motion detection + janela deslizante
2. Pré-classificação automática (YOLOv8n COCO + pose score)
3. Interface interativa de revisão manual
4. Exportação dos segmentos rotulados para `data/client_labeled/`

> Os dados do cliente ficam **completamente isolados** em `data/client_labeled/`.  
> Nunca são misturados com datasets públicos neste notebook.

In [ ]:
!pip install -q ultralytics opencv-python-headless pandas matplotlib ipywidgets tqdm openpyxl Pillow

In [ ]:
import re, cv2, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from datetime import timedelta
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
from ultralytics import YOLO
from PIL import Image as PILImage

warnings.filterwarnings("ignore")

# Modelo de pose YOLOv8n-pose (substitui MediaPipe — sem dependência de versão)
# Baixado automaticamente pelo ultralytics na primeira execução (~6 MB)
_pose_model = None

def _get_pose_model():
    global _pose_model
    if _pose_model is None:
        _pose_model = YOLO("yolov8n-pose.pt")
    return _pose_model

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CLIENT_VIDEO_DIR  = Path("../data/client_videos")
ANALYSIS_DIR      = Path("../data/client_analysis")   # saída do nb01
LABELED_DIR       = Path("../data/client_labeled")    # ISOLADO — só dados do cliente

SEGMENT_SEC       = 8      # duração de cada clipe
OVERLAP_SEC       = 2      # sobreposição
MOTION_THRESHOLD  = 15.0
MIN_ACTIVE_FRAC   = 0.20   # % mínimo de frames ativos para manter segmento

VIOLENCE_THRESH   = 0.30   # score YOLO para violence
POSE_THRESH       = 0.42   # score de pose para pre_violence
AUTO_CONF_MIN     = 0.68   # abaixo → revisão manual
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

for d in [LABELED_DIR/"violence", LABELED_DIR/"non_violence"]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dados cliente (isolados): {LABELED_DIR.resolve()}")

## 1. Carregar catálogo do notebook 01

In [ ]:
cat_json = ANALYSIS_DIR / "catalog.json"
assert cat_json.exists(), "Execute o notebook 01_client_eda.ipynb primeiro."

df_cat = pd.read_json(cat_json)
df_vid = df_cat[(df_cat["readable"]==True) & (df_cat["is_image"]==False)].copy()
print(f"Vídeos para segmentar: {len(df_vid)}")

with open(ANALYSIS_DIR/"motion_profiles.json") as f:
    motion_cache = json.load(f)
print(f"Perfis de movimento carregados: {len(motion_cache)}")

## 2. Segmentação — motion + janela deslizante

In [ ]:
def sliding_windows(duration, seg=SEGMENT_SEC, overlap=OVERLAP_SEC):
    step, wins = seg - overlap, []
    t = 0.0
    while t + seg <= duration + 0.5:
        wins.append((t, min(t + seg, duration)))
        t += step
    return wins


def filter_by_motion(windows, motion_scores, motion_times, fps_orig):
    """Mantém apenas janelas onde pelo menos MIN_ACTIVE_FRAC dos frames têm movimento."""
    if not motion_scores:
        return windows
    times  = np.array(motion_times)
    scores = np.array(motion_scores)
    result = []
    for (ws, we) in windows:
        mask = (times >= ws) & (times < we)
        if mask.sum() == 0: continue
        active = (scores[mask] > MOTION_THRESHOLD).mean()
        if active >= MIN_ACTIVE_FRAC:
            result.append((ws, we))
    return result


all_segs = []
for _, row in tqdm(df_vid.iterrows(), total=len(df_vid), desc="Segmentando"):
    dur  = row.get("duration_sec") or 0
    if dur < SEGMENT_SEC: continue
    wins = sliding_windows(dur)
    mdata = motion_cache.get(row["filename"], {})
    wins = filter_by_motion(wins, mdata.get("scores",[]), mdata.get("times",[]),
                            row.get("fps",25) or 25)
    for i, (ts, te) in enumerate(wins):
        all_segs.append(dict(
            source_file=row["filename"], source_path=row["filepath"],
            local=row.get("local"), camera=row.get("camera"),
            tag=row.get("tag"), t_start=round(ts,2), t_end=round(te,2),
            duration=round(te-ts,2),
            seg_id=f"{Path(row['filepath']).stem}_seg{i:04d}",
            auto_label=None, label_conf=None, final_label=None, reviewed=False,
        ))

df_seg = pd.DataFrame(all_segs)
print(f"\n✓ Segmentos com movimento: {len(df_seg)}")
if "local" in df_seg.columns:
    display(df_seg.groupby("local").size().reset_index(name="segmentos"))

## 3. Pré-classificação automática

In [ ]:
# Usa modelo treinado se disponível, senão YOLOv8n COCO
TRAINED = Path("../models/best.pt")
yolo    = YOLO(str(TRAINED) if TRAINED.exists() else "yolov8n.pt")
USE_TRAINED = TRAINED.exists()
print(f"Modelo: {'treinado' if USE_TRAINED else 'YOLOv8n COCO (sem classes de violência)'}");
if not USE_TRAINED:
    print("  Score de violência baseado apenas em postura (MediaPipe).")


# Índices COCO keypoints usados pelo YOLOv8-pose:
# 0=nose 5=l_shoulder 6=r_shoulder 7=l_elbow 8=r_elbow
# 9=l_wrist 10=r_wrist 11=l_hip 12=r_hip
def aggression_score(frame):
    """Estima score de postura agressiva via YOLOv8n-pose (substitui MediaPipe)."""
    pose_model = _get_pose_model()
    results = pose_model(frame, verbose=False, conf=0.3)
    best = 0.0
    for r in results:
        if r.keypoints is None or r.keypoints.xy is None:
            continue
        kps = r.keypoints.xy.cpu().numpy()   # [N_persons, 17, 2] — coords em pixels
        if len(kps) == 0:
            continue
        h, w = frame.shape[:2]
        for kp in kps:  # per person
            # Normaliza para [0,1]
            kpn = kp.copy().astype(float)
            kpn[:, 0] /= max(w, 1)
            kpn[:, 1] /= max(h, 1)
            def pt(i): return kpn[i]  # [x, y] normalizado
            try:
                # Braços levantados acima dos ombros
                sh_y = (pt(5)[1] + pt(6)[1]) / 2
                arm  = min(1.0, (max(0, sh_y - pt(9)[1]) +
                                  max(0, sh_y - pt(10)[1])) * 2)
                # Inclinação do torso (nose vs. quadril)
                hip_c = (pt(11) + pt(12)) / 2
                vec   = pt(0) - hip_c
                torso = min(1.0, abs(np.arctan2(vec[0], -vec[1])) / (np.pi / 3))
                # Distância pulso-pulso (mãos abertas/levantadas)
                spd   = min(1.0, np.linalg.norm(pt(9) - pt(10)) * 2)
                score = float(np.clip(0.4*arm + 0.3*torso + 0.3*spd, 0, 1))
                best  = max(best, score)
            except Exception:
                continue
    return best


def classify_segment(src_path, t_start, t_end, n=8):
    cap = cv2.VideoCapture(src_path)
    if not cap.isOpened(): return dict(auto_label="non_violence",label_conf=0.5,v=0.0,pv=0.0,frames=[])
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25
    f0,f1 = int(t_start*fps), int(t_end*fps)
    idxs  = [int(f0+i*(f1-f0)/n) for i in range(n)]
    vs,pvs,frames = [],[],[]
    for fi in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret: continue
        preds = yolo(frame, verbose=False, conf=0.25)[0]
        vc = pvc = 0.0
        for box in preds.boxes:
            cls, conf = int(box.cls[0]), float(box.conf[0])
            if USE_TRAINED:
                if cls == 1: vc  = max(vc,  conf)
                if cls == 2: pvc = max(pvc, conf)
        pvc = max(pvc, aggression_score(frame))
        vs.append(vc); pvs.append(pvc)
        frames.append(cv2.cvtColor(cv2.resize(frame,(280,158)),cv2.COLOR_BGR2RGB))
    cap.release()
    if not vs: return dict(auto_label="non_violence",label_conf=0.5,v=0.0,pv=0.0,frames=[])
    vm, pvm = max(vs), max(pvs)
    if vm >= VIOLENCE_THRESH:    label, conf = "violence",     vm
    elif pvm >= POSE_THRESH:     label, conf = "violence",     pvm*0.75
    else:                        label, conf = "non_violence", 1.0-max(vm,pvm)
    return dict(auto_label=label, label_conf=round(conf,3),
                v=round(vm,3), pv=round(pvm,3), frames=frames)


frames_cache = {}
MAX_SEGS = None   # None = todos
seg_list = df_seg.to_dict("records")[:MAX_SEGS] if MAX_SEGS else df_seg.to_dict("records")

for seg in tqdm(seg_list, desc="Classificando"):
    r = classify_segment(seg["source_path"], seg["t_start"], seg["t_end"])
    seg.update(auto_label=r["auto_label"], label_conf=r["label_conf"],
               v_score=r["v"], pv_score=r["pv"], final_label=r["auto_label"])
    frames_cache[seg["seg_id"]] = r["frames"]

df_seg = pd.DataFrame(seg_list)
n_v  = (df_seg["auto_label"]=="violence").sum()
n_nv = (df_seg["auto_label"]=="non_violence").sum()
n_uc = (df_seg["label_conf"] < AUTO_CONF_MIN).sum()
print(f"\n  violence     : {n_v} ({n_v/len(df_seg)*100:.1f}%)")
print(f"  non_violence : {n_nv} ({n_nv/len(df_seg)*100:.1f}%)")
print(f"  incertos     : {n_uc} → revisão manual")

## 4. Revisão manual dos casos incertos

In [ ]:
to_review = df_seg[df_seg["label_conf"] < AUTO_CONF_MIN].copy()
print(f"Segmentos para revisão: {len(to_review)}")

if len(to_review) == 0:
    print("✓ Nenhuma revisão necessária.")
else:
    state = {"idx": 0, "ids": to_review.index.tolist()}
    out   = widgets.Output()

    def redraw():
        out.clear_output(wait=True)
        with out:
            if state["idx"] >= len(state["ids"]):
                print("✓ Revisão concluída!"); return
            ridx = state["ids"][state["idx"]]
            seg  = df_seg.loc[ridx]
            frms = frames_cache.get(seg["seg_id"], [])
            cur, tot = state["idx"]+1, len(state["ids"])
            display(HTML(f"""
            <div style='background:#f8f8f8;border:1px solid #ccc;padding:10px;
                 border-radius:6px;font-family:monospace;font-size:13px'>
            <b>Revisão {cur}/{tot}</b><br>
            Arquivo : {seg['source_file']}<br>
            Local   : {seg.get('local','?')} &nbsp; Câmera: {seg.get('camera','?')}<br>
            Trecho  : {timedelta(seconds=int(seg['t_start']))} → {timedelta(seconds=int(seg['t_end']))}
            &nbsp;({seg['duration']:.1f}s)<br>
            Sugerido: <b style='color:{"red" if seg["auto_label"]=="violence" else "green"}'>
                {seg['auto_label'].upper()}</b>
            &nbsp; conf={seg['label_conf']:.2f} &nbsp; v={seg['v_score']:.2f} &nbsp; pv={seg['pv_score']:.2f}
            </div>"""))
            if frms:
                n = min(len(frms),4)
                fig, axes = plt.subplots(1,n,figsize=(n*3.5,2.8))
                if n==1: axes=[axes]
                for ax,fr in zip(axes,frms[:n]): ax.imshow(fr); ax.axis("off")
                plt.tight_layout(pad=0.2); plt.show()

    def make_btn(label, style, fl):
        b = widgets.Button(description=label, button_style=style,
                           layout=widgets.Layout(width="155px",height="36px"))
        def click(_):
            ridx = state["ids"][state["idx"]]
            df_seg.at[ridx,"final_label"] = fl
            df_seg.at[ridx,"reviewed"]    = True
            state["idx"] += 1
            pb.value = state["idx"]
            redraw()
        b.on_click(click); return b

    pb = widgets.IntProgress(value=0,min=0,max=len(state["ids"]),
                              description="Revisão:",
                              layout=widgets.Layout(width="380px"))
    redraw()
    display(widgets.VBox([
        pb,
        widgets.HBox([make_btn("Violência","danger","violence"),
                      make_btn("Sem violência","success","non_violence"),
                      make_btn("Pular","warning",None)]),
        out
    ]))

## 5. Exportar segmentos rotulados (isolado — só cliente)

In [ ]:
df_seg["final_label"] = df_seg.apply(
    lambda r: r["final_label"] if pd.notna(r.get("final_label")) else r["auto_label"], axis=1)
df_final = df_seg[df_seg["final_label"].notna()].copy()


def export_clip(src, t0, t1, out_path):
    cap = cv2.VideoCapture(src)
    if not cap.isOpened(): return False
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t0*fps))
    wri = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w,h))
    for _ in range(int((t1-t0)*fps)):
        ret,fr = cap.read()
        if not ret: break
        wri.write(fr)
    cap.release(); wri.release()
    return out_path.exists() and out_path.stat().st_size > 0


log = []
for _, seg in tqdm(df_final.iterrows(), total=len(df_final), desc="Exportando"):
    lbl = seg["final_label"]
    safe = re.sub(r"[^\w-]","_",seg["seg_id"])
    out  = LABELED_DIR / lbl / f"{seg.get('local','X')}_{seg.get('camera','X')}_{safe}.mp4"
    if out.exists():
        log.append({**seg.to_dict(), "status":"exists"}); continue
    ok = export_clip(seg["source_path"], seg["t_start"], seg["t_end"], out)
    log.append({**seg.to_dict(), "status":"ok" if ok else "error"})

df_log = pd.DataFrame(log)
df_log.drop(columns=["source_path"],errors="ignore").to_csv(
    ANALYSIS_DIR/"segments_labeled.csv", index=False)

nv  = len(list((LABELED_DIR/"violence").glob("*.mp4")))
nnv = len(list((LABELED_DIR/"non_violence").glob("*.mp4")))
print(f"\n✓ Exportados (isolados em client_labeled/)")
print(f"  violence     : {nv}")
print(f"  non_violence : {nnv}")
print(f"  Log          : {ANALYSIS_DIR/'segments_labeled.csv'}")
print("\nPróximo: 03_dataset_client_only.ipynb")